# Stage 5 · Reasoning & Test-Time Compute — SOLUTION
### Topics: Best-of-N · RLVR · DAPO Fixes · Entropy Collapse · Length Hacking · Training Diagnostics


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import re
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass, field


---
## 1 · Best-of-N Sampling & Test-Time Compute Scaling

Instead of (or in addition to) better training, we can spend more compute at **inference time**.

### Best-of-N (BoN / rejection sampling)
1. Sample N independent completions from the policy
2. Score each with a reward model or verifiable function
3. Return the highest-scoring one

**Key scaling result (Snell et al., 2024):**
Performance scales as **O(log N)** — diminishing returns but consistent gains.

### Weighted Best-of-N (WBoN)
Instead of hard argmax, weight samples by exponentiated reward:
$$w_i \propto \pi_{ref}(y_i|x) \cdot \exp(r(x,y_i)/\beta)$$

This is the **optimal RLHF policy** — WBoN at inference *approximates* what RLHF training achieves.

### BoN vs Beam Search
| | BoN | Beam Search |
|---|---|---|
| Diversity | High (independent) | Low (pruned tree) |
| Parallelisable | Yes | Partially |
| RM dependent | Yes | Optional |
| Optimal | No | Locally |


In [ ]:
def best_of_n(
    completions: List[str],
    rewards:     torch.Tensor,   # (N,)
) -> Tuple[str, int, float]:
    """Return (best_completion, best_index, best_reward)."""
    best_idx = rewards.argmax().item()
    return completions[best_idx], best_idx, rewards[best_idx].item()


def weighted_best_of_n_weights(
    log_probs_ref: torch.Tensor,  # (N,) — log pi_ref(y|x)
    rewards:       torch.Tensor,  # (N,)
    beta:          float = 1.0,
) -> torch.Tensor:                # (N,) — normalised sampling weights
    """
    log w_i = log_pi_ref(y_i) + r_i / beta
    Normalise via softmax.
    Low beta → concentrates on best reward.
    High beta → approaches reference distribution (ignores reward).
    """
    log_weights = log_probs_ref + rewards / beta
    return torch.softmax(log_weights, dim=0)


def simulate_bon_scaling(
    reward_fn,             # callable () -> float, draws one reward sample
    n_values:  List[int],
    n_trials:  int = 500,
    seed:      int = 0,
) -> Dict[int, float]:
    """
    For each N: run n_trials experiments, each sampling N rewards and taking max.
    Returns {N: mean_max_reward}.
    """
    np.random.seed(seed)
    results = {}
    for N in n_values:
        max_rewards = [max(reward_fn() for _ in range(N)) for _ in range(n_trials)]
        results[N] = float(np.mean(max_rewards))
    return results


def bon_marginal_efficiency(bon_rewards: Dict[int, float]) -> Dict[int, float]:
    """
    Marginal gain per additional sample vs N=1:
    efficiency(N) = (r_N - r_1) / (N - 1)
    """
    r1 = bon_rewards.get(1, 0.0)
    return {N: (r - r1) / max(N - 1, 1) for N, r in bon_rewards.items()}


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(42)
N = 10
completions = [f"completion_{i}" for i in range(N)]
rewards     = torch.randn(N)

best_c, best_i, best_r = best_of_n(completions, rewards)
assert best_r == rewards.max().item(), "Must return the max reward"
assert completions[best_i] == best_c

# WBoN: weights sum to 1, all non-negative
log_p_ref = torch.randn(N)
w = weighted_best_of_n_weights(log_p_ref, rewards, beta=1.0)
assert abs(w.sum().item() - 1.0) < 1e-5
assert (w >= 0).all()

# Low beta -> concentrated; high beta -> diffuse
w_lo = weighted_best_of_n_weights(log_p_ref, rewards, beta=0.01)
w_hi = weighted_best_of_n_weights(log_p_ref, rewards, beta=100.0)
assert w_lo.max() > w_hi.max(), "Low beta should concentrate weight on best completion"

# BoN scaling: larger N -> higher expected max reward (Gaussian draws)
scaling = simulate_bon_scaling(lambda: float(np.random.randn()), [1, 4, 16, 64], n_trials=300)
assert scaling[64] > scaling[16] > scaling[4] > scaling[1], "BoN should improve monotonically with N"

eff = bon_marginal_efficiency(scaling)
print(f"best_of_n               ✓  best_r={best_r:.4f}")
print(f"weighted_best_of_n      ✓  weight sum={w.sum().item():.5f}")
print(f"simulate_bon_scaling    ✓  N=1:{scaling[1]:.3f}  N=4:{scaling[4]:.3f}  N=16:{scaling[16]:.3f}  N=64:{scaling[64]:.3f}")
print(f"bon_marginal_efficiency ✓  efficiency drops with N (diminishing returns)")


---
## 2 · RLVR — RL with Verifiable Rewards

**RLVR** is the training paradigm behind DeepSeek-R1 and similar reasoning models.  
Replace the learned reward model with a **deterministic verifier** — the reward cannot be hacked.

### The RLVR loop
```
for each batch of (prompt, ground_truth) pairs:
    1. Sample G completions per prompt (e.g., G=8)
    2. Verify each: reward = verifier(completion, ground_truth)  ← 0 or 1
    3. Compute group-relative advantages (GRPO)
    4. Update policy with clipped surrogate + KL penalty
```

### Cold-start problem
At the start of training, the model rarely gets correct answers → all rewards = 0  
→ all advantages = 0 → **no gradient signal**.

**Solutions:**
1. **SFT warm-up:** supervised fine-tune on worked examples before RL
2. **Curriculum:** start with easy problems, increase difficulty
3. **Rejection sampling fine-tuning (RFT):** generate many candidates, keep correct ones, SFT on them
4. **Format reward:** give partial credit just for using `<think>...</think>` tags

### The DeepSeek-R1 emergent CoT finding
With *only* verifiable rewards and no supervised CoT data, models spontaneously develop:
- Self-correction: backtracking and revising
- Exploration: trying multiple approaches
- Reflection: "Wait, let me reconsider..."


In [ ]:
def rlvr_reward(
    completion:   str,
    ground_truth: str,
    format_bonus: float = 0.1,
) -> float:
    """
    Extract final number ('#### N') from both strings, compare.
    Returns 1.0 if correct, 0.0 if wrong.
    Adds format_bonus if <think>...</think> tags are present in order.
    """
    def extract_answer(text: str) -> Optional[float]:
        for pat in [r'####\s*([+-]?[\d,.]+)', r'=\s*([+-]?[\d,.]+)', r'([+-]?[\d.]+)$']:
            m = re.search(pat, text.strip())
            if m:
                try:
                    return float(m.group(1).replace(',', ''))
                except ValueError:
                    continue
        return None

    pred  = extract_answer(completion)
    truth = extract_answer(ground_truth)

    if pred is None or truth is None:
        return 0.0

    correct    = abs(pred - truth) < 1e-5
    has_format = ('<think>' in completion and '</think>' in completion
                  and completion.index('<think>') < completion.index('</think>'))

    return float(correct) + (format_bonus if has_format else 0.0)


def grpo_advantages_for_rlvr(
    rewards: torch.Tensor,   # (B*G,)
    G:       int,
    eps:     float = 1e-8,
) -> torch.Tensor:           # (B*G,)
    """
    Group-relative normalisation.
    Special case: if all rewards in a group are identical, return zeros (no gradient).
    """
    rg    = rewards.view(-1, G)                                    # (B, G)
    mean  = rg.mean(dim=1, keepdim=True)
    std   = rg.std(dim=1, keepdim=True)
    # If std ≈ 0 (all same reward), advantages are meaningless → return 0
    mask  = (std > eps).float()
    adv   = mask * (rg - mean) / (std + eps)
    return adv.view(-1)


def rejection_sampling_filter(
    completions:  List[str],
    ground_truths: List[str],
    threshold:    float = 1.0,
) -> List[Tuple[str, str]]:
    """
    RFT data collection: keep only (completion, prompt) pairs where
    rlvr_reward >= threshold.
    Returns [(prompt_placeholder, correct_completion)] — for SFT training.
    """
    kept = []
    for comp, gt in zip(completions, ground_truths):
        if rlvr_reward(comp, gt) >= threshold:
            kept.append((gt, comp))   # (ground_truth used as prompt proxy)
    return kept


# ── Sanity checks ─────────────────────────────────────────────────────────
# Reward function
cot_correct  = "<think>Let me compute 2+2=4</think> #### 4"
bare_correct = "#### 4"
wrong        = "#### 5"
no_number    = "I think the answer might be four"

assert rlvr_reward(cot_correct,  "#### 4") == 1.1, f"CoT+correct should be 1.1, got {rlvr_reward(cot_correct, '#### 4')}"
assert rlvr_reward(bare_correct, "#### 4") == 1.0
assert rlvr_reward(wrong,        "#### 4") == 0.0
assert rlvr_reward(no_number,    "#### 4") == 0.0

# GRPO advantages: all-same reward -> all-zero advantage
B, G = 3, 4
rewards_same = torch.zeros(B * G)
adv_same = grpo_advantages_for_rlvr(rewards_same, G)
assert adv_same.abs().max() < 1e-5, "Identical rewards → zero advantages"

# Normal case: each group normalised to ~mean 0, std 1
rewards_varied = torch.randn(B * G)
adv_varied = grpo_advantages_for_rlvr(rewards_varied, G)
for i in range(B):
    g = adv_varied[i*G:(i+1)*G]
    assert g.mean().abs() < 1e-4, f"Group {i} mean not zero: {g.mean()}"

# RFT filter
comps  = ["<think>2+2=4</think> #### 4", "#### 5", "<think>3+1=4</think> #### 4"]
truths = ["#### 4", "#### 4", "#### 4"]
kept   = rejection_sampling_filter(comps, truths, threshold=1.0)
assert len(kept) == 2, f"Expected 2 correct completions, got {len(kept)}"

print(f"rlvr_reward              ✓  cot={rlvr_reward(cot_correct,'#### 4'):.1f}  bare={rlvr_reward(bare_correct,'#### 4'):.1f}  wrong={rlvr_reward(wrong,'#### 4'):.1f}")
print(f"grpo_advantages_for_rlvr ✓  all-same→zeros, varied→normalised")
print(f"rejection_sampling_filter ✓  kept {len(kept)}/3 correct completions")


---
## 3 · DAPO — Four Fixes for GRPO at Scale (ByteDance, 2025)

DAPO identifies and resolves four failure modes that appear when scaling GRPO to hard reasoning tasks.

### Fix 1: Token-level loss normalisation
GRPO averages loss over sequences → long sequences contribute less per-token.  
**DAPO:** normalise loss by **total completion tokens** across the batch, not by sequences.

### Fix 2: Clip-higher (asymmetric clipping)
Standard PPO clips both increases and decreases equally.  
For sparse verifiable rewards, clipping ratio *increases* on positive advantages is too conservative.  
**DAPO:** use $\varepsilon_{high} > \varepsilon_{low}$:
$$\text{clip}(\rho, 1 - \varepsilon_{low},\ 1 + \varepsilon_{high\ \text{if}\  A>0\ \text{else}\ low})$$

### Fix 3: Entropy bonus to prevent collapse
Add $-\alpha \cdot H(\pi)$ directly to the loss.  
Since $H$ is differentiable, this creates a gradient that pushes the policy toward higher entropy.

### Fix 4: Overlong sequence filtering
Models learn to generate very long responses (reward hacking via length).  
**DAPO:** filter sequences exceeding `max_length`; replace their rewards with a penalty.


In [ ]:
def token_level_policy_loss(
    logits:     torch.Tensor,   # (B, T, V)
    input_ids:  torch.Tensor,   # (B, T)
    advantages: torch.Tensor,   # (B,)  per-sequence, detached
    comp_mask:  torch.Tensor,   # (B, T) — 1 for completion tokens
    logits_old: torch.Tensor,   # (B, T, V)  detached
    epsilon:    float = 0.2,
) -> torch.Tensor:
    """
    DAPO token-level loss: ratio and advantage computed per-token.
    Normalise by total completion tokens (not sequences).
    """
    lp_new = F.log_softmax(logits,              dim=-1)   # (B,T,V)
    lp_old = F.log_softmax(logits_old.detach(), dim=-1)

    # Gather log-prob of the actual token
    tok_lp_new = lp_new.gather(2, input_ids.unsqueeze(-1)).squeeze(-1)  # (B,T)
    tok_lp_old = lp_old.gather(2, input_ids.unsqueeze(-1)).squeeze(-1)

    ratio     = torch.exp(tok_lp_new - tok_lp_old)
    adv_tok   = advantages.unsqueeze(-1).expand_as(ratio)                # (B,T)

    surr1     = ratio * adv_tok
    surr2     = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * adv_tok
    token_loss = -torch.min(surr1, surr2)                                # (B,T)

    # DAPO fix: divide by TOTAL completion tokens, not number of sequences
    n_comp = comp_mask.float().sum().clamp(min=1)
    return (token_loss * comp_mask.float()).sum() / n_comp


def dapo_clip_loss(
    log_probs_new: torch.Tensor,   # (B,)
    log_probs_old: torch.Tensor,   # (B,)  detached
    advantages:    torch.Tensor,   # (B,)  detached
    eps_low:       float = 0.2,
    eps_high:      float = 0.28,
) -> torch.Tensor:
    """
    Asymmetric clip: higher epsilon for positive advantages.
    This allows larger policy updates on good completions.
    """
    ratio = torch.exp(log_probs_new - log_probs_old.detach())
    lo    = 1.0 - eps_low
    hi    = torch.where(advantages > 0,
                        torch.full_like(advantages, 1.0 + eps_high),
                        torch.full_like(advantages, 1.0 + eps_low))
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, lo, hi) * advantages
    return -torch.min(surr1, surr2).mean()


def entropy_from_logits(
    logits:    torch.Tensor,   # (B, T, V)
    comp_mask: torch.Tensor,   # (B, T)
) -> torch.Tensor:             # ()  scalar — mean over completion tokens
    """H_t = -sum_v p(v) log p(v). Averaged over completion tokens only."""
    lp    = F.log_softmax(logits, dim=-1)       # (B,T,V)
    p     = lp.exp()
    H_t   = -(p * lp).sum(dim=-1)              # (B,T)
    n_comp = comp_mask.float().sum().clamp(min=1)
    return (H_t * comp_mask.float()).sum() / n_comp


def filter_overlong(
    rewards:    torch.Tensor,   # (B,)
    lengths:    torch.Tensor,   # (B,)
    max_length: int   = 4096,
    penalty:    float = -1.0,
) -> torch.Tensor:
    """Replace reward with penalty for sequences exceeding max_length."""
    return torch.where(lengths > max_length,
                       torch.full_like(rewards, penalty),
                       rewards)


def dapo_total_loss(
    logits:        torch.Tensor,   # (B, T, V)
    input_ids:     torch.Tensor,   # (B, T)
    logits_old:    torch.Tensor,   # (B, T, V)  detached
    log_probs_seq: torch.Tensor,   # (B,)  per-sequence new logprobs
    log_probs_old_seq: torch.Tensor,  # (B,)  detached
    advantages:    torch.Tensor,   # (B,)  detached
    comp_mask:     torch.Tensor,   # (B, T)
    eps_low:    float = 0.2,
    eps_high:   float = 0.28,
    ent_coef:   float = 0.01,
) -> Tuple[torch.Tensor, Dict]:
    """
    Full DAPO loss:
      L = token_level_policy_loss (clip-higher, token-normalised)
        - ent_coef * entropy     (entropy bonus)
    """
    pg_loss  = token_level_policy_loss(logits, input_ids, advantages, comp_mask, logits_old, eps_low)
    entropy  = entropy_from_logits(logits, comp_mask)
    total    = pg_loss - ent_coef * entropy
    return total, {"pg_loss": pg_loss.item(), "entropy": entropy.item()}


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, V = 4, 12, 50
prompt_len = 4
logits     = torch.randn(B, T, V, requires_grad=True)
input_ids  = torch.randint(0, V, (B, T))
comp_mask  = torch.cat([torch.zeros(B,prompt_len), torch.ones(B,T-prompt_len)], dim=1).long()
adv        = torch.randn(B)
logits_old = logits.detach().clone()

# Token-level loss
tok_loss = token_level_policy_loss(logits, input_ids, adv, comp_mask, logits_old)
assert tok_loss.shape == ()

# DAPO clip: asymmetric
lp_new = torch.randn(B, requires_grad=True)
lp_old = lp_new.detach()
loss_dapo = dapo_clip_loss(lp_new, lp_old, adv, eps_low=0.2, eps_high=0.28)
assert loss_dapo.shape == ()

# Entropy: in (0, log(V)]
ent = entropy_from_logits(logits, comp_mask)
assert 0 < ent.item() <= math.log(V) + 1e-4, f"Entropy out of range: {ent.item()}"

# Overlong filter
rewards = torch.ones(B)
lengths = torch.tensor([100, 200, 5000, 300])
filtered = filter_overlong(rewards, lengths, max_length=4096)
assert filtered[2].item() == -1.0 and filtered[0].item() == 1.0

# Full DAPO loss + gradient
total, info = dapo_total_loss(logits, input_ids, logits_old, lp_new, lp_old, adv, comp_mask)
total.backward()
assert logits.grad is not None

print(f"token_level_policy_loss ✓  loss={tok_loss.item():.4f}")
print(f"dapo_clip_loss          ✓  loss={loss_dapo.item():.4f}")
print(f"entropy_from_logits     ✓  H={ent.item():.4f}  (log(V)={math.log(V):.3f})")
print(f"filter_overlong         ✓  {filtered.tolist()}")
print(f"dapo_total_loss         ✓  {info}")


---
## 4 · Simulating Entropy Collapse

Entropy collapse is the most common silent failure in RLVR training at scale.

### What happens step by step
1. Policy gradient pushes mass toward high-reward tokens
2. High-probability tokens dominate; entropy drops
3. The model explores less → discovers fewer correct solutions
4. Reward signal becomes even sparser → gradient nearly zero
5. Model is stuck — can't improve

### The entropy-exploitation tradeoff
| Entropy | Exploration | Exploitation |
|---|---|---|
| High (uniform) | Maximum | None |
| Medium | Balanced | Balanced |
| Low (peaked) | None | Maximum |
| Collapsed (→0) | None | Wrong solution |

### Monitoring thresholds (practical)
- **Warning:** entropy < 0.5 nats (for vocab size V=50k, max entropy ≈ 10.8 nats)
- **Critical:** entropy < 0.1 nats — essentially deterministic
- **Healthy range:** 0.5×H_max to 0.9×H_max


In [ ]:
def simulate_entropy_dynamics(
    n_steps:     int   = 120,
    ent_coef:    float = 0.0,
    reward_temp: float = 1.0,   # higher = sharper reward gradient
    V:           int   = 30,    # vocabulary size
    seed:        int   = 42,
) -> Dict[str, List[float]]:
    """
    Simulate policy entropy over training steps.
    Model: logits over V tokens; RL loss pushes P(token_0) high.
    ent_coef adds entropy bonus to resist collapse.
    Returns dict with entropy and p_best (prob of greedy token) histories.
    """
    torch.manual_seed(seed)
    logits = torch.zeros(V, requires_grad=True)
    opt    = torch.optim.SGD([logits], lr=0.15)
    history = {"entropy": [], "p_best": [], "gradient_norm": []}

    for _ in range(n_steps):
        probs    = F.softmax(logits, dim=0)
        log_probs = F.log_softmax(logits, dim=0)
        entropy  = -(probs * log_probs).sum()

        # RL signal: reward token 0 (simulates sparse correct-answer reward)
        rl_loss  = -reward_temp * log_probs[0]

        # Entropy bonus (DAPO fix)
        loss = rl_loss - ent_coef * entropy

        opt.zero_grad()
        loss.backward()
        grad_norm = logits.grad.norm().item()
        opt.step()

        with torch.no_grad():
            history["entropy"].append(entropy.item())
            history["p_best"].append(probs[0].item())
            history["gradient_norm"].append(grad_norm)

    return history


def detect_entropy_collapse(
    entropy_history: List[float],
    window:          int   = 10,
    threshold:       float = 0.3,
) -> Tuple[bool, int]:
    """
    Detect if entropy has collapsed: rolling mean < threshold sustained for `window` steps.
    Returns (collapsed: bool, collapse_step: int or -1).
    """
    for i in range(window, len(entropy_history) + 1):
        window_mean = np.mean(entropy_history[i - window: i])
        if window_mean < threshold:
            return True, i - window
    return False, -1


# ── Run comparison ─────────────────────────────────────────────────────────
coefs = [0.0, 0.02, 0.1]
sims  = {c: simulate_entropy_dynamics(n_steps=100, ent_coef=c) for c in coefs}

print("Entropy dynamics (start → end):")
for c, hist in sims.items():
    h   = hist["entropy"]
    collapsed, step = detect_entropy_collapse(h, window=5, threshold=0.2)
    print(f"  ent_coef={c:.2f}: H={h[0]:.3f} → {h[-1]:.3f}  "
          f"collapsed={collapsed}" + (f" at step {step}" if collapsed else ""))

# Verify: no entropy bonus → collapse; large bonus → stable
assert sims[0.0]["entropy"][-1] < 0.2, "No entropy bonus should lead to collapse"
assert sims[0.1]["entropy"][-1] > sims[0.0]["entropy"][-1], "Entropy bonus should resist collapse"
print("simulate_entropy_dynamics ✓  detect_entropy_collapse ✓")


---
## 5 · End-to-End Training Monitoring Dashboard

Everything you need to monitor a RLVR/GRPO training run at scale.

### Minimum monitoring checklist

| Metric | Healthy range | Action if unhealthy |
|---|---|---|
| `reward_mean` | Increasing steadily | — |
| `reward_nonzero_frac` | > 5% | Curriculum; SFT warm-up |
| `kl_from_ref` | < 5 nats | Increase β; reduce lr |
| `entropy` | > 0.5 | Increase `ent_coef` |
| `clip_fraction` | 0.1–0.3 | Reduce lr if > 0.5 |
| `advantage_std` | ≈ 1.0 | Check normalisation |
| `length_mean` | Stable | Add overlong filter if increasing |

### The KL–Reward Pareto frontier
A well-trained model maximises reward while minimising KL.  
Plot `kl_from_ref` vs `reward_mean` to see if KL is "buying" proportional reward gains.

### Length hacking signature
```
reward_mean ↑ (proxy reward increasing)
length_mean ↑ (completions getting longer)
reward_nonzero_frac stable (same fraction correct, just longer)
```
This pattern → model is hacking length, not learning to reason better.


In [ ]:
@dataclass
class StepMetrics:
    step:                    int
    reward_mean:             float
    reward_std:              float
    reward_nonzero_frac:     float
    kl_from_ref:             float
    entropy:                 float
    clip_fraction:           float
    advantage_std:           float
    length_mean:             float
    length_std:              float


def compute_step_metrics(
    rewards:       torch.Tensor,   # (B,)
    log_probs_new: torch.Tensor,   # (B,)
    log_probs_old: torch.Tensor,   # (B,)
    log_probs_ref: torch.Tensor,   # (B,)
    advantages:    torch.Tensor,   # (B,)
    lengths:       torch.Tensor,   # (B,)
    entropy:       float,
    step:          int,
    epsilon:       float = 0.2,
) -> StepMetrics:
    ratio     = torch.exp(log_probs_new - log_probs_old.detach())
    clip_frac = ((ratio < 1-epsilon) | (ratio > 1+epsilon)).float().mean().item()
    kl        = (log_probs_new - log_probs_ref.detach()).mean().item()
    return StepMetrics(
        step                = step,
        reward_mean         = rewards.mean().item(),
        reward_std          = rewards.std().item(),
        reward_nonzero_frac = (rewards > 0).float().mean().item(),
        kl_from_ref         = kl,
        entropy             = entropy,
        clip_fraction       = clip_frac,
        advantage_std       = advantages.std().item(),
        length_mean         = lengths.float().mean().item(),
        length_std          = lengths.float().std().item(),
    )


def health_check(m: StepMetrics) -> List[str]:
    """Return warning strings for any metric outside healthy range."""
    warns = []
    if m.entropy < 0.5:
        warns.append(f"ENTROPY COLLAPSE: {m.entropy:.3f} < 0.5 — increase ent_coef")
    if m.kl_from_ref > 5.0:
        warns.append(f"KL HIGH: {m.kl_from_ref:.3f} > 5 — increase beta / reduce lr")
    if m.reward_nonzero_frac < 0.05:
        warns.append(f"SPARSE REWARD: {m.reward_nonzero_frac:.1%} — cold start? use curriculum")
    if m.clip_fraction > 0.5:
        warns.append(f"HIGH CLIP: {m.clip_fraction:.3f} — learning rate too high")
    if not (0.5 < m.advantage_std < 3.0):
        warns.append(f"ADV STD: {m.advantage_std:.3f} — check GRPO normalisation")
    return warns


def detect_length_hacking(
    history: List[StepMetrics],
    window:  int = 10,
) -> bool:
    """
    Length hacking signature: reward_mean increasing AND length_mean increasing
    while reward_nonzero_frac is NOT increasing (same % correct, just longer).
    Check over the last `window` steps.
    """
    if len(history) < window * 2:
        return False
    early  = history[-window*2:-window]
    recent = history[-window:]
    reward_increasing = np.mean([m.reward_mean for m in recent]) > np.mean([m.reward_mean for m in early])
    length_increasing = np.mean([m.length_mean for m in recent]) > np.mean([m.length_mean for m in early]) * 1.2
    correct_stable    = abs(np.mean([m.reward_nonzero_frac for m in recent]) -
                            np.mean([m.reward_nonzero_frac for m in early])) < 0.05
    return reward_increasing and length_increasing and correct_stable


# ── Simulate and monitor ───────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
history: List[StepMetrics] = []

# Steps 0–14: healthy training
for step in range(15):
    m = compute_step_metrics(
        rewards       = torch.rand(B) * 0.5 + step * 0.01,
        log_probs_new = torch.randn(B) * 0.1,
        log_probs_old = torch.randn(B) * 0.1,
        log_probs_ref = torch.randn(B) * 0.1,
        advantages    = torch.randn(B),
        lengths       = torch.randint(100, 300, (B,)),
        entropy       = 1.5 - step * 0.03,
        step          = step,
    )
    history.append(m)
    warns = health_check(m)
    if warns:
        print(f"  Step {step:2d} WARNINGS: {warns}")

# Steps 15–24: length hacking
for step in range(15, 25):
    m = compute_step_metrics(
        rewards       = torch.rand(B) * 0.5 + 0.15 + (step-15) * 0.02,
        log_probs_new = torch.randn(B) * 0.1,
        log_probs_old = torch.randn(B) * 0.1,
        log_probs_ref = torch.randn(B) * 0.1,
        advantages    = torch.randn(B),
        lengths       = torch.randint(200 + (step-15)*100, 400 + (step-15)*100, (B,)),
        entropy       = 1.0,
        step          = step,
    )
    history.append(m)

hacking = detect_length_hacking(history, window=5)
print(f"\nTraining run completed: {len(history)} steps")
print(f"detect_length_hacking: {hacking}")
assert hacking, "Should detect length hacking in the simulated run"
print("compute_step_metrics ✓  health_check ✓  detect_length_hacking ✓")
